# Lab 0-03: Compare Three Models on PII Extraction

In this warm-up notebook, you will use three class models from the same local Ollama service and compare how they extract PII and device identifiers from a short synthetic case note.

This task is a good fit for Lab 0 because it is:
- easy to run
- relevant to digital forensics
- simple enough to compare by observation before you revise a prompt
- based on synthetic data rather than real personal data

## Follow the Story

Imagine that three AI assistants are reading the same short case note. Each assistant uses a different large language model (LLM):

- Assistant 1 uses `qwen3.5:0.8b`
- Assistant 2 uses `qwen3.5:27b`
- Assistant 3 uses `gemma4:e4b`

Their shared task is to read the same synthetic forensic case note and find five kinds of information: names, phone numbers, email addresses, physical addresses, and device IDs such as an IMEI or serial number. Each assistant must return its findings in the same JSON format.

The notebook gives all three assistants the same instructions, saves what each one says, and helps you look at the answers side by side.

### Goal

The goal is to show that different LLMs can give different answers to the same forensic task. After giving all three AI assistants the same case note, you will compare:

- what each model found
- whether it followed the requested JSON format
- how the answers differ
- how long each model took

This prepares you to improve prompts later and choose an appropriate model for a task.

The story moves in this order:

1. Check that the notebook is in the right folder and can read its settings.
2. See which models are ready to use.
3. Give the same case note and the same question to three models.
4. Keep each answer, including how long it took.
5. Put a few useful facts from the answers into a short checklist.

The notebook does not choose a winner for you. Its job is to gather the evidence so that you can decide what you notice.

## Step 0: Check Your Setup

Run the next four short code blocks in order. Each one checks one part of the local setup: the repository, the lab folder, the test model, and the Ollama address.

This step checks the local setup only. It does not contact Ollama yet; Step 1 does that when it asks which models are available.

### 0.1 Locate the Repository

This first block brings in the Python tools used later and finds the repository folder that contains this lab.

In [1]:
import json
from html import escape
from pathlib import Path
from time import perf_counter

import requests
from dotenv import dotenv_values
from openai import OpenAI

LAB_NAME = 'lab0_03_llm_api_and_model_basics'
lab_dir = Path.cwd().resolve()
repo_root = lab_dir.parent

print('Repo root:', repo_root)


Repo root: /home/frank/projects/agentic-AI4-forensics


### 0.2 Check the Lab Folder

The notebook must be opened from `lab0_03_llm_api_and_model_basics` so it can find this lab's settings file.

In [2]:
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(f'Open this notebook from the {LAB_NAME} folder.')

print('Lab folder:', lab_dir)


Lab folder: /home/frank/projects/agentic-AI4-forensics/lab0_03_llm_api_and_model_basics


### 0.3 Read the Test Model Setting

This block checks that `.env.example` and your private `.env` file exist, then reads the test model name from `.env`. The test model confirms that the settings file was read; it is not one of the three models compared later.

In [3]:
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError('Expected .env in this folder. Copy .env.example to .env first.')

config = dotenv_values(env_path)
default_model = config.get('MODEL')
if not default_model:
    raise ValueError("MODEL is missing from this lab's .env")

print("Test model from this lab's .env:", default_model)


Test model from this lab's .env: qwen3:8b


### 0.4 Read the Ollama Address

Finally, this block reads the Ollama address from `.env` and creates the connection object used in the next steps. It still does not contact Ollama; the model roll call in Step 1 is the first real contact.

In [4]:
ollama_base_url = config.get('OLLAMA_BASE_URL')
if not ollama_base_url:
    raise ValueError("OLLAMA_BASE_URL is missing from this lab's .env")

client = OpenAI(base_url=ollama_base_url, api_key='ollama')
print('Ollama address:', ollama_base_url)


Ollama address: http://localhost:11434/v1


## Step 1: Discover Available Models

Run the next code block to take a roll call of the models installed on your computer. It also shows their sizes, which helps explain why some models may respond more slowly than others.

In [5]:
# WHAT THIS BLOCK DOES: Ask the local model service which models are ready.
# It saves their names in one list and their sizes in another easy-to-use
# lookup, then prints a friendly roll call for the class.
tags_url = ollama_base_url.rstrip('/').replace('/v1', '/api/tags')

response = requests.get(tags_url, timeout=10)
response.raise_for_status()

models_data = response.json().get('models', [])
available_models = []
model_size_lookup = {}

print('Available models:')
for item in models_data:
    model_name = item.get('name')
    if not model_name:
        continue

    available_models.append(model_name)
    size_bytes = item.get('size')
    if isinstance(size_bytes, (int, float)):
        size_text = f'{size_bytes / (1024 ** 3):.2f} GB'
    else:
        size_text = 'size unavailable'

    model_size_lookup[model_name] = size_text
    print(f'{len(available_models)}. {model_name} ({size_text})')

if len(available_models) < 3:
    raise ValueError('This homework needs at least 3 available models.')


Available models:
1. qwen3-vl:latest (5.72 GB)
2. kimi-k2.5:cloud (0.00 GB)
3. kimi-k2.6:cloud (0.00 GB)
4. qwen3.6:latest (22.29 GB)
5. gemma4:31b (18.50 GB)
6. gemma4:e4b (8.95 GB)
7. huihui_ai/qwen3.5-abliterated:latest (16.22 GB)
8. qwen3.5:2b (2.55 GB)
9. qwen3.5:9b (6.14 GB)
10. qwen3.5:0.8b (0.96 GB)
11. qwen3.5:35b-a3b (22.23 GB)
12. qwen3:8b (4.87 GB)
13. qwen3.5:27b (16.22 GB)
14. llama3.3:70b (39.60 GB)


## Step 2: Confirm Three Comparison Models

Use these three models for the comparison so everyone in the class works with the same set:
- `qwen3.5:0.8b`
- `qwen3.5:27b`
- `gemma4:e4b`

The next code block checks that all three are available from Ollama before the comparison begins. This keeps everyone in the class working with the same three assistants.

In [6]:
# WHAT THIS BLOCK DOES: Choose the three assistants for this activity.
# It stops early if one is missing or if the same model was listed twice,
# because a fair comparison needs three different models.
models_to_compare = ['qwen3.5:0.8b', 'qwen3.5:27b', 'gemma4:e4b']

missing_models = [name for name in models_to_compare if name not in available_models]

print('Models selected for comparison:')
for model_name in models_to_compare:
    size_text = model_size_lookup.get(model_name, 'size unavailable')
    print(f'- {model_name} ({size_text})')

if missing_models:
    raise ValueError(f'These required models are missing from Ollama: {missing_models}')

if len(set(models_to_compare)) != 3:
    raise ValueError('Choose 3 different models before continuing.')


Models selected for comparison:
- qwen3.5:0.8b (0.96 GB)
- qwen3.5:27b (16.22 GB)
- gemma4:e4b (8.95 GB)


## Step 3: Synthetic Case Note

Use the same made-up case note for all three models. The next code block simply stores that note in a variable and prints it, so you can see exactly what every model will read.

In [7]:
# WHAT THIS BLOCK DOES: Keep one shared case note for the whole activity.
# Because every model reads these exact same words, differences in the
# answers come from the models, not from different evidence.
case_note = """
Case note:
Investigator Maya Chen documented an interview with Jordan Lee at 2458 West Pine Street, Springfield, IL 62704.
Jordan Lee said a suspicious text came from 415-555-0187 and referenced alex.romero88@example.com.
A second contact for follow-up was Priya Nair at 202-555-0142 and priyanair@sample.org.
The seized phone record listed IMEI 356938035643809 and serial number SN-A19XZ-4421.
""".strip()

print(case_note)

Case note:
Investigator Maya Chen documented an interview with Jordan Lee at 2458 West Pine Street, Springfield, IL 62704.
Jordan Lee said a suspicious text came from 415-555-0187 and referenced alex.romero88@example.com.
A second contact for follow-up was Priya Nair at 202-555-0142 and priyanair@sample.org.
The seized phone record listed IMEI 356938035643809 and serial number SN-A19XZ-4421.


## Step 4: Run the Model Comparison

### 4.1 Create the Shared Extraction Prompt

All three models should answer the **same prompt** so you can compare them fairly. The next block joins the instructions and the case note into one message that will be reused without changes.

In [8]:
# WHAT THIS BLOCK DOES: Write one shared question for every model.
# It asks for JSON, a simple label-and-list format, so you can compare not
# only what each model finds but also whether it follows the requested format.
comparison_prompt = f"""
Extract PII from the synthetic case note below.

Return valid JSON only.
Use exactly these keys:
- names
- phone_numbers
- email_addresses
- physical_addresses
- device_ids

Rules:
- Keep each item exactly as it appears in the note.
- Use arrays for all five keys.
- Do not include explanations.
- Do not add keys beyond the five listed above.

Case note:
{case_note}
""".strip()

print(comparison_prompt)

Extract PII from the synthetic case note below.

Return valid JSON only.
Use exactly these keys:
- names
- phone_numbers
- email_addresses
- physical_addresses
- device_ids

Rules:
- Keep each item exactly as it appears in the note.
- Use arrays for all five keys.
- Do not include explanations.
- Do not add keys beyond the five listed above.

Case note:
Case note:
Investigator Maya Chen documented an interview with Jordan Lee at 2458 West Pine Street, Springfield, IL 62704.
Jordan Lee said a suspicious text came from 415-555-0187 and referenced alex.romero88@example.com.
A second contact for follow-up was Priya Nair at 202-555-0142 and priyanair@sample.org.
The seized phone record listed IMEI 356938035643809 and serial number SN-A19XZ-4421.


### 4.2 Ask All Three Models

This code block has two small helpers and then runs the shared question three times. The first helper tidies an answer before checking whether it is JSON. The second asks one model the question, records how long it waited, and saves the answer. Finally, the loop repeats that work for all three models.


In [9]:
def clean_json_text(text: str) -> str:
    # A model may put its answer in a code box or add extra words.
    # This helper looks for the part between { and }, where JSON usually is.
    cleaned = text.strip()
    if cleaned.startswith('```'):
        cleaned = cleaned.strip('`')
        cleaned = cleaned.replace('json\n', '', 1).strip()
    start = cleaned.find('{')
    end = cleaned.rfind('}')
    if start != -1 and end != -1 and end > start:
        cleaned = cleaned[start:end + 1]
    return cleaned

def ask_model(model_name: str, prompt: str) -> dict:
    # Ask one model the shared question and time how long it takes to answer.
    start = perf_counter()
    response = client.chat.completions.create(
        model=model_name,
        messages=[{'role': 'user', 'content': prompt}]
    )
    elapsed = perf_counter() - start
    # Keep the original answer for reading later, plus a tidied copy for the
    # JSON check below.
    raw_text = response.choices[0].message.content
    cleaned_text = clean_json_text(raw_text)

    # If JSON cannot be read, remember that fact instead of stopping.
    # A model not following the format is useful evidence for comparison.
    try:
        parsed = json.loads(cleaned_text)
        parse_error = None
    except Exception as exc:
        parsed = None
        parse_error = str(exc)

    return {
        'model': model_name,
        'seconds': round(elapsed, 2),
        'raw_text': raw_text,
        'parsed': parsed,
        'parse_error': parse_error,
    }

# Ask each selected model the same question and keep the results together.
results = []
for model_name in models_to_compare:
    print(f'Running {model_name}...')
    results.append(ask_model(model_name, comparison_prompt))

print('Comparison complete.')


Running qwen3.5:0.8b...
Running qwen3.5:27b...
Running gemma4:e4b...
Comparison complete.


### 4.3 Read the Full Answers

Run this block to read each answer exactly as the model gave it. Do not worry if the answers look different. Spotting differences in wording, missing details, or formatting is part of the activity.


In [10]:
# WHAT THIS BLOCK DOES: Print each answer exactly as it was received.
# Read these full answers before relying on the short checklist below. An
# answer can follow the JSON format and still miss or change a detail.
for result in results:
    print('=' * 80)
    print('Model:', result['model'])
    print('Time (seconds):', result['seconds'])
    print('Parse error:', result['parse_error'])
    print('-' * 80)
    print(result['raw_text'])
    print()

Model: qwen3.5:0.8b
Time (seconds): 72.17
Parse error: None
--------------------------------------------------------------------------------
{
    "names": ["Investigator Maya Chen", "Jordan Lee", "Priya Nair"],
    "phone_numbers": ["415-555-0187", "202-555-0142"],
    "email_addresses": ["alex.romero88@example.com"],
    "physical_addresses": ["2458 West Pine Street, Springfield, IL 62704"],
    "device_ids": ["356938035643809", "SN-A19XZ-4421"]
}

Model: qwen3.5:27b
Time (seconds): 133.37
Parse error: None
--------------------------------------------------------------------------------
{
  "names": [
    "Maya Chen",
    "Jordan Lee",
    "Priya Nair"
  ],
  "phone_numbers": [
    "415-555-0187",
    "202-555-0142"
  ],
  "email_addresses": [
    "alex.romero88@example.com",
    "priyanair@sample.org"
  ],
  "physical_addresses": [
    "2458 West Pine Street, Springfield, IL 62704"
  ],
  "device_ids": [
    "356938035643809",
    "SN-A19XZ-4421"
  ]
}

Model: gemma4:e4b
Time (secon

### 4.4 Compare the Results in a Table

The full answers above are important, but they can be hard to compare at a glance. This step puts the key details from all three answers into one table.

For each model, the table shows:
- response time in seconds
- whether the response followed the requested JSON format
- the names, contacts, addresses, and device IDs it found

In [11]:
# WHAT THIS BLOCK DOES: Turn each long answer into one row in a comparison table.
# The table is a quick guide, not a score and not a final judgment about quality.
comparison_summary = []

for result in results:
    # Use the answer as a label-and-list dictionary only when the JSON check worked.
    parsed = result['parsed'] if isinstance(result['parsed'], dict) else None
    keys_found = sorted(parsed.keys()) if isinstance(parsed, dict) else []
    device_ids_found = parsed.get('device_ids', []) if isinstance(parsed, dict) else []

    comparison_summary.append({
        'model': result['model'],
        'seconds': result['seconds'],
        'valid_json': parsed is not None,
        'keys_found': keys_found,
        'names_found': parsed.get('names', []) if parsed else [],
        'phone_numbers_found': parsed.get('phone_numbers', []) if parsed else [],
        'email_addresses_found': parsed.get('email_addresses', []) if parsed else [],
        'physical_addresses_found': parsed.get('physical_addresses', []) if parsed else [],
        'device_ids_found': device_ids_found,
    })

# Turn a list into text that fits neatly inside one table cell.
def table_items(items):
    return '<br>'.join(escape(str(item)) for item in items) if items else '—'

table_rows = []
for summary in comparison_summary:
    json_status = 'Yes' if summary['valid_json'] else 'No'
    table_rows.append(f"""
        <tr>
          <td>{escape(summary['model'])}</td>
          <td>{summary['seconds']}</td>
          <td>{json_status}</td>
          <td>{table_items(summary['names_found'])}</td>
          <td>{table_items(summary['phone_numbers_found'])}</td>
          <td>{table_items(summary['email_addresses_found'])}</td>
          <td>{table_items(summary['physical_addresses_found'])}</td>
          <td>{table_items(summary['device_ids_found'])}</td>
        </tr>
    """)

from IPython.display import HTML, display
table_html = """
<table style='border-collapse: collapse'>
  <tr>
    <th>Model</th><th>Seconds</th><th>Valid JSON?</th><th>Names</th>
    <th>Phone numbers</th><th>Email addresses</th><th>Physical addresses</th><th>Device IDs</th>
  </tr>
  {rows}
</table>
""".format(rows=''.join(table_rows))

display(HTML(table_html))

Model,Seconds,Valid JSON?,Names,Phone numbers,Email addresses,Physical addresses,Device IDs
qwen3.5:0.8b,72.17,Yes,Investigator Maya ChenJordan LeePriya Nair,415-555-0187202-555-0142,alex.romero88@example.com,"2458 West Pine Street, Springfield, IL 62704",356938035643809SN-A19XZ-4421
qwen3.5:27b,133.37,Yes,Maya ChenJordan LeePriya Nair,415-555-0187202-555-0142,alex.romero88@example.compriyanair@sample.org,"2458 West Pine Street, Springfield, IL 62704",356938035643809SN-A19XZ-4421
gemma4:e4b,18.25,Yes,Maya ChenJordan LeePriya Nair,415-555-0187202-555-0142,alex.romero88@example.compriyanair@sample.org,"2458 West Pine Street, Springfield, IL 62704",IMEI 356938035643809SN-A19XZ-4421


## Step 5: Reflection Questions

Replace this text with short answers to the questions below.

Use the summary above and the raw outputs to support your answers.

1. Which model was the fastest?
2. Which models returned valid JSON?
3. Did the three models use the same `device_ids` format? Describe one difference you noticed.
4. Which response was easiest to read and why?
5. What is one prompt rule you might add before doing `04_prompt_revision_assignment.ipynb`?

## Step 6: Submission

Save the notebook with:
- the discovered model list
- the three selected models
- the raw outputs from all three models
- the simple comparison summary
- your short reflection